In [43]:
from simular import PyEvm, PyAbi, Contract, create_account, contract_from_inline_abi
import numpy as np
import pandas as pd

In [44]:
NODE_URL = 'https://rpc.berachain.com'
evm = PyEvm.from_fork(url = NODE_URL)

In [45]:
def create_contract(evm, bytecode=None, abi_file_path='', address=''):
    with open(abi_file_path) as f:
        abi = f.read()
    if bytecode is None:
        abi = PyAbi.from_abi_bytecode(abi, bytecode)
        return Contract(evm, abi).at(address)
    else:
        abi = PyAbi.from_abi_bytecode(abi, bytes.fromhex(bytecode))
        return Contract(evm, abi)

In [46]:
BERABORROW_DenManagerGetters = '0xFA7908287c1f1B256831c812c7194cb95BB440e6'

den_manager_getter = create_contract(evm, bytecode=None, abi_file_path="abis/BeraborrowDenManagerGetters.abi", address=BERABORROW_DenManagerGetters)

In [47]:
collaterals_info = den_manager_getter.getAllCollateralsAndDenManagers.call()

In [48]:
symbols = []
collaterals = []
price = []
tvl = []

for collateral in range(len(collaterals_info)):
    token_address = collaterals_info[collateral][0]
    den_manager_address = collaterals_info[collateral][1][0] 

    den_manager = create_contract(evm, bytecode=None, abi_file_path="abis/BeraborrowPermissionedDenManager.abi", address=den_manager_address)
    token = create_contract(evm, bytecode=None, abi_file_path="abis/WETH.abi", address=token_address)

    scale = 10**token.decimals.call()

    symbols.append(token.symbol.call())
    collaterals.append(den_manager.getEntireSystemColl.call()/scale)
    price.append(den_manager.fetchPrice.call()/10**18)
    tvl.append(den_manager.getEntireSystemColl.call()/scale*den_manager.fetchPrice.call()/10**18)

df = pd.DataFrame({
    'token': symbols,
    'token amount': collaterals,
    'price (in USD)': price,
    'TVL': tvl
})
df = df.sort_values(by=df.columns[-1], ascending=False).reset_index(drop=True)

print(df)

                      token  token amount  price (in USD)           TVL
0                BB.PUMPBTC  2.180040e+03    1.040999e+05  2.269419e+08
1            BB.SOLVBTC.BBN  9.718149e+02    1.063790e+05  1.033807e+08
2                 BB.UNIBTC  4.752296e+02    1.036035e+05  4.923544e+07
3                BB.SOLVBTC  4.000493e+02    1.062801e+05  4.251730e+07
4                   BB.WETH  7.251758e+02    2.623853e+03  1.902755e+06
5             BB.kWETH-WBTC  4.211405e-04    3.370671e+09  1.419526e+06
6   BB.kSOLVBTC-SOLVBTC.BBN  5.626771e+01    1.447070e+04  8.142330e+05
7                   BB.WBTC  6.199460e+00    1.066354e+05  6.610816e+05
8                    BBiBGT  3.306719e+04    5.573339e+00  1.842947e+05
9            BB.kWBTC-WBERA  2.703670e-03    5.554937e+07  1.501872e+05
10          BB.kWBERA-iBERA  3.530436e+05    4.048976e-01  1.429465e+05
11               BB.BERAETH  2.658943e+01    2.641279e+03  7.023011e+04
12                  BB.WETH  2.193112e+01    2.623853e+03  5.754